## Predicting-mortality-of-icu-patients-the-physionet-computing-in-cardiology-challenge-2012-1.0.0
### author--Yuyao Lei
### netid--yl1216

## AI-use statement
I used ChatGPT as supplementary aids during the development and documentation of this project in the following ways:
- Code Development and Debugging
  - Assisted in drafting and reviewing Python code for data preprocessing, feature construction, merging predictor data with outcome data, and checking data consistency.
  - Helped identify potential coding errors, resolve implementation issues, and improve the organization and reproducibility of the analysis workflow.
- Statistical Modeling
  - Provided guidance on implementing and understanding LASSO logistic regression, spline logistic regression, and random forest models.
  - Assisted with explanations of cross-validation, hyperparameter tuning, and the use of AUROC to compare predictive performance.
- Model Evaluation and Interpretation
  - Helped organize cross-validation results and explain model performance metrics in the code.
  - Assisted in understanding feature-importance plots and partial-effect plots, including identifying limitations and potential issues that require further investigation.
- Documentation and Scientific Writing
  - Helped improve the clarity and organization of the written report.
  - Assisted in explaining methodological decisions, and drafting descriptions of the modeling procedures and results.

I reviewed
and tested all suggested code, verified the outputs independently, and
made the final decisions regarding data processing, predictors, models,
and interpretation. All results were checked
against the assignment requirements and the data. I ensured that I understand the code and analysis presented, and I take full responsibility for all submitted work.

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import missingno as msno
from tableone import TableOne

import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import expit, logit
from scipy.stats import norm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc,
    confusion_matrix
)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
#code: setup
ROOT=Path.cwd()
#an assertion that "checkpoint1.ipynb" is in it
assert(ROOT/"checkpoint1.ipynb").exists(),"checkpoint1.ipynb not found"
DATA=ROOT/"data"
OUT=ROOT/"output"
OUT.mkdir(exist_ok=True)

In [ ]:
#code: check data
def check_set_a(data_dir: Path):
    files=list((data_dir/"set-a").glob("*"))
    assert len(files)>0,"no file in set-a"
    return len(files)

n_files=check_set_a(DATA)
assert n_files==4000, f"expected 4000 records, but only {n_files} found"


In [ ]:
#code: load
def load_set_a(data_dir: Path):
    data_dir=Path(data_dir)
    all_long=[]
    #iterate through 4000 txt file records
    for file in data_dir.glob("*.txt"):
        #set the name of txt as sample_id
        sample_id=file.stem
        #read as csv
        df=pd.read_csv(file)
        df["id"]=sample_id
        all_long.append(df)
    long_df=pd.concat(all_long,ignore_index=True)
    wide_df=(long_df.pivot_table(index=["id","Time"],columns="Parameter",values="Value",aggfunc="mean").reset_index())
    return long_df, wide_df
long_df,wide_df=load_set_a(DATA/"set-a")
#check the structure of data frame
wide_df["id"] = wide_df["id"].astype(str)
long_df["id"] = long_df["id"].astype(str)
#make id str type, or error would occur when merging the data frame in table1
print(long_df.shape)
print(wide_df.shape)
print(long_df)
print(wide_df)
long_df.to_csv(OUT/ "set-a_long.csv",index=False)
wide_df.to_csv(OUT/ "set-a_wide.csv",index=False)


## What exists at prediction time
At prediction time, **early hospitalization information exists** and **future information (survival or death) is not yet observed**.

Information including:
- patient identifier (id)
- measured time (Time)
- basic demographic variables (e.g., age, gender)
- vital signs (e.g., HR, Temp)
- lab measurements (e.g., Urine, Creatinine)
- other recorded parameters

In long format, the following columns are included in the structure:
- id: patient identifier
- time: time since admission
- parameter: measured variable name
- value: measurement value

In wide formats, the first row includes every parameter name: 
- id
- Time
- other measurements
and the values of each variable are following after the column names.

**Only information available up to a fixed prediction time window (<47:36) is included.** Measurements occurring after this window are excluded to **avoid data leakage**.
Ensure that the prediction simulates a realistic clinical decision-making scenario.

## Outcome summary

In [ ]:
#code: load outcomes
#from pathlib import Path
#print(list(ROOT.rglob("Outcomes*")))
#print((DATA / "Outcomes-a.txt").exists())
outcomes_raw=pd.read_csv(DATA / "Outcomes-a.txt")
#Question when debuging: if there isn't a space in fornt of "Outcomes-a.txt", error occurs, but why?
outcomes_df=outcomes_raw[["RecordID","In-hospital_death"]].rename(columns={"RecordID":"id","In-hospital_death":"death"})
outcomes_df["id"] = outcomes_df["id"].astype(str)
#让id都是字符型数据
print(outcomes_df.head())
#outcome summary
print(outcomes_df["death"].value_counts())
print(outcomes_df["death"].mean())

In [ ]:
# load outcome for analyze SAPS-T & SOFA
outcome2_df=outcomes_raw[["RecordID","SAPS-I","SOFA","Length_of_stay","Survival","In-hospital_death"]].rename(columns={"In-hospital_death":"death"})

The variables include:
- RecordID
- SAPS-I
- SOFA
- Length_of_stay
- Survival
- In-hospital_death

We only extract **RecordID** as **id** and **In-hospital_death** as **death**. The status of patients is recorded as **death=1, survival=0**. 

Print the **counts of deaths and survivals**, **the mean** of outcome summary shows the **proportion of deaths in addmission patients**.

The outcome label should not be available at prediction time.


## Table1: Grouped by in-hospital death
The table summarizes patient characteristics and lab measurements, grouped by in-hospital death.

In [ ]:
#Table1
wide_df_clean = wide_df.drop(columns=["Time"], errors="ignore")#missingmap in id order
wide_df_clean2=wide_df.drop(columns=["id"], errors="ignore")#missingmap in time order
#if column["Time"] not be dropped, warning occurs because the variable Time is not numeric.
table1_df = wide_df_clean.merge(outcomes_df, on="id", how="left")
columns = [c for c in table1_df.columns if c != "death"]
Table1 = TableOne(
    table1_df,
    columns=columns,
    categorical=[],   
    groupby="death",
    pval=True
)
Table1

### key observation
It reports:
- mean (SD) for each variable
- missing counts
- group-wise comparison
- *P*-value for statistical significance
### interpretation
- The distribution of several variables differs between the death and survival groups (e.g., AST, Creatinine), suggesting potential associations with patient outcomes.
- Variables with statistically significant *P*-values (p < 0.05) may indicate important predictors of in-hospital mortality, although these should be interpreted cautiously due to multiple comparisons and potential confounding.
- Some variables (e.g., MechVent) may show unstable or undefined *P*-values due to limited variation or small effective sample sizes.
- The magnitude of differences (in addition to *P*-values) should be considered when assessing clinical relevance.


## Nullity matrix
This nullity matrix visualizes the presence (black) and absence (white) of measurements across variables for all observations.
Rows are id of patient in order, and each column represents a variable.
Overall, the nullity matrix reveals substantial and structured missingness across variables, indicating that missingness is likely informative rather than random.

In [ ]:
#missingness map
msno.matrix(wide_df_clean, figsize=(8, 3.0), fontsize=10, sparkline=False) 
plt.title("nullity matrix: rows in admission id order, white = missing") 
plt.tight_layout() 
plt.show()

### key observation
- Many lab variables exhibit substantial missingness.
- Some variables are consistently measured across observations (e.g., HR, Weight).
- The missingness pattern appears structured rather than random.
- Clusters of missingness across variables suggest that certain tests are ordered together, reflecting underlying clinical workflows.
### interpretation
- measurements are not consistently collected across patients, given that missingness is common in some variables.
- consistently measured variables suggest that they are part of routine monitoring.
- The observed missingness patterns suggest that the data are unlikely to be Missing Completely At Random (MCAR). Instead, missingness may depend on patient condition or clinical decisions, which has important implications for downstream modeling.

## Missingness informative
The number of times each lab was measured predicts in-hospital death.
Before looking at a single lab value: clinicians order a test when they are worried, and their worry is informative.
This is why “focus on what is observed, not what is missing”. The Informed Presence is also a leakage trap: the lab measurement was ordered because someone already suspected the diagnosis.

In [ ]:
#missingness informative
count_df = (
    long_df
    .groupby(["id", "Parameter"])
    .size()
    .reset_index(name="n_measurements")
)
count_wide = count_df.pivot(
    index="id",
    columns="Parameter",
    values="n_measurements"
).fillna(0)
long_df["id"] = long_df["id"].astype(str)
outcomes_df["id"] = outcomes_df["id"].astype(str)
#unify the type of id in different data frames, or error occurs when merge
plot_df = count_wide.merge(outcomes_df, on="id", how="left")
parameters = count_wide.columns

for p in parameters:
    temp = plot_df[["death", p]].copy()
    temp.columns = ["death", "count"]

    plt.figure(figsize=(5, 3))
    sns.boxplot(data=temp, x="death", y="count")

    plt.title(f"{p}: informative presence")
    plt.xlabel("death (0=survived, 1=died)")
    plt.ylabel("number of measurements")

    plt.tight_layout()
    plt.show()

### key observation
- The boxplot shows the distribution of the number of each measurements per patient, stratified by in-hospital death (0 = survived, 1 = died).
- The y-axis represents the number of times each measurement was measured for each patient.
- For some measurements (e.g., ALT, ALP), patients who died tend to have more frequent measurements compared to those who survived.
- For some measurements (e.g., AST, Albumin), a large proportion of patients have zero measurements, indicating that this test is not routinely performed for all patients.
- For some measurements (e.g., SysABP, WBC), the frequency of measurement did not differ from between survived and died group; the presence of some measurements is not such informative (e.g., FiO2). 
- The death group often shows more extreme values (e.g., cholesterol, Creatinine), suggesting that some patients underwent repeated testing, possibly due to clinical deterioration.
### interpretation
- Measurement frequency is likely driven by clinical decisions, where sicker patients are monitored more frequently.
- For informative presence, missingness is unlikely to be random and may reflect underlying patient severity (i.e., Missing Not At Random, MNAR).
- The number of measurements itself may carry predictive information, independent of the measured values.
- If the boxplot indicates that measurement frequency differs by outcome, suggesting that missingness is informative and related to patient severity.

## Check duplication

In [ ]:
# check duplication
print(
    "wide_df duplicated RecordID:",
    wide_df["id"].duplicated().sum()
)
print(
    "wide_df duplicated RecordID:",
    wide_df["RecordID"].duplicated().sum()
)
print(
    "outcome_df duplicated RecordID:",
    outcomes_df["id"].duplicated().sum()
)

print("\nMost frequent RecordID:")
print(wide_df["RecordID"].value_counts().head(10))
print("\nMost frequent ID:")
print(wide_df["id"].value_counts().head(10))

# one-to-one relationship between id and RecordID 
print("\nFirst 15 rows:")
display(wide_df[["id", "RecordID", "Time"]].head(15))

# no missing rows in RecordID
print("\nRows where RecordID is not missing:")
display(
    wide_df.loc[
        wide_df["RecordID"].notna(),
        ["id", "RecordID", "Time"]
    ].head(15)
)



NameError: name 'wide_df' is not defined

- For "RecordID", all counts returned by value_counts().head(10) are equal to 1;
- For "id", multiple repeated values exhibits varying frequencies. 

This suggests that the apparent duplication may be driven by missing values rather than representing genuine duplicate records.

The result shows, "id" has 4,000 unique values and is complete, with no missingness. Given that each "id" corresponds to multiple time points, it is the correct ICU stay identifier for this wide-format dataset.

## Build modeling dataframe

In [ ]:
# verify “id” in wide_df consistent with outcomes_df["id"]
wide_ids = pd.to_numeric(
    wide_df["id"], errors="coerce"
).dropna().astype("int64")

outcomes_ids = pd.to_numeric(
    outcomes_df["id"], errors="coerce"
).dropna().astype("int64")

wide_id_set = set(wide_ids.unique())
outcomes_id_set = set(outcomes_ids.unique())

print("the number of ICU stay in wide_df:", len(wide_id_set))
print("the number of ICU stay in outcomes_df:", len(outcomes_id_set))
print("the number of id in wide_df, but not in outcomes_df:",
      len(wide_id_set - outcomes_id_set))
print("the number of id in outcomes_df, but not in wide_df:",
      len(outcomes_id_set - wide_id_set))

print("\nwhether duplicated id in outcomes_df:")
print(not outcomes_df["id"].duplicated().any())

The results showed “id” in wide_df are consistent with “id” in outcomes_df.

In [ ]:
# select the record in the first 48 hours
# copy wide_df
df = wide_df.copy()

# standardize time
time_parts = df["Time"].astype(str).str.split(":", expand=True)

df["_hours"] = (
    pd.to_numeric(time_parts[0], errors="coerce")
    + pd.to_numeric(time_parts[1], errors="coerce") / 60
)

# in the first 48 hours
df = df.loc[
    df["_hours"].between(0, 48, inclusive="both")
].copy()

print("records in 0–48 hours:", len(df))
print("ICU stay in 0–48 hours:", df["id"].nunique())

# columns which are not treated as measured variable
exclude_cols = {"id", "RecordID", "Time", "_hours"}

measurement_cols = [
    col for col in df.columns
    if col not in exclude_cols
]

# choosing static and categorical variable
static_cols = [
    col for col in ["Age", "Gender", "Height", "ICUType"]
    if col in measurement_cols
]

# others are treated as numeric variables
continuous_cols = [
    col for col in measurement_cols
    if col not in static_cols
]

# check numeric
print("\nnon-numeric:")
print(df[continuous_cols].select_dtypes(exclude="number").columns.tolist())

# smmarize feacture (mean/min/max/last/count) within ICU stay
feature_df = (
    df.groupby("id")[continuous_cols]
      .agg(["mean", "min", "max", "last", "count"])
)

feature_df.columns = [
    f"{variable}_{stat}"
    for variable, stat in feature_df.columns
]

# selecting static and categorical variable
if static_cols:
    static_df = df.groupby("id")[static_cols].first()
    feature_df = feature_df.join(static_df)

# urine feature
if "Urine" in continuous_cols:
    urine_df = df.groupby("id")["Urine"].sum(min_count=1).rename(
        "Urine_sum"
    )
    feature_df = feature_df.join(urine_df)

# unify id as RecordID
feature_df = feature_df.reset_index().rename(
    columns={"id": "RecordID"}
)

print("\nfeacture table shape:", feature_df.shape)
print("ICU stay:", feature_df["RecordID"].nunique())
print("whether RecordID unique:",
      not feature_df["RecordID"].duplicated().any())

display(feature_df.head())

- Predictors are derived exclusively from measurements collected during the first 48 hours.
- For repeated measurements, summary features include the mean, minimum, maximum, last observed value, and number of non-missing measurements. The measurement count features also describe the observation process and may capture clinically informative differences in monitoring intensity.
- For non-repeated measurements, RecordID is retained only as an identifier. Length_of_stay and Survival are excluded because they describe information beyond the prediction window.
- For benchmark scores, SAPS-I and SOFA are not used as predictors in the three fitted models. 
- For categorical variables, ICUType and Gender are treated as categorical variables rather than numeric quantities.

In [ ]:
# merge with outcomes_df
# unify RecordID
feature_df["RecordID"] = pd.to_numeric(
    feature_df["RecordID"], errors="raise"
).astype("int64")

outcomes = outcomes_df.copy()
outcomes = outcomes.rename(columns={"id":"RecordID"})

outcomes["RecordID"] = pd.to_numeric(
    outcomes["RecordID"], errors="raise"
).astype("int64")

# check duplication in outcomes_df 
assert not outcomes["RecordID"].duplicated().any(), (
    "duplicated RecordID in outcomes_df."
)

# one-to-one merge
model_df = feature_df.merge(
    outcomes,
    on="RecordID",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("shape after merge:", model_df.shape)
print("ICU stay after merge:", model_df["RecordID"].nunique())

print("\nmatching:")
print(model_df["_merge"].value_counts())

# missingness in death
if "In-hospital_death" in model_df.columns:
    print(
        "\nIn-hospital_death missingness:",
        model_df["In-hospital_death"].isna().sum()
    )

# check duplicated ICU stay 
assert not model_df["RecordID"].duplicated().any()

# delete redundant columns
model_df = model_df.drop(columns="_merge")

# final model_df
display(model_df.head())

The modeling table is checked for duplicate identifiers, missing outcomes, and class imbalance before model fitting.
### Key observation
- The merged dataset contains 4,000 ICU stays and 193 columns.
- All 4,000 records matched successfully during the merge; no unmatched records were found.
- There are substantial variation in missingness across clinical measurements, with some variables having no observations and others having multiple measurements.
### Interpretation
- The successful one-to-one matching supports the consistency of the merge between the feature and outcome datasets.
- Missing clinical measurements may reflect differences in data availability across patients and should be assessed to avoid leakage.

## Predictor selection

In [ ]:
# check outcome
y1=model_df["death"]
model_df["death"].value_counts()
model_df["death"].value_counts(normalize=True)

# check one row per stay
model_df["RecordID"].duplicated().sum()

os.makedirs("output", exist_ok=True)
# Confirm that the outcome exists
TARGET = "death"
assert TARGET in model_df.columns, (
    f"Cannot find outcome column {TARGET!r}. "
    f"Available columns: {model_df.columns.tolist()}"
)

# One row per ICU stay
assert "RecordID" in model_df.columns
assert model_df["RecordID"].notna().all()
assert not model_df["RecordID"].duplicated().any()

# Remove rows without a known outcome; report how many were removed
n_before = len(model_df)
analysis_df = model_df.loc[model_df[TARGET].notna()].copy()
n_missing_outcome = n_before - len(analysis_df)

# Outcome must be binary 0/1
analysis_df[TARGET] = pd.to_numeric(
    analysis_df[TARGET], errors="raise"
).astype(int)

assert set(analysis_df[TARGET].unique()).issubset({0, 1}), (
    "The mortality outcome must be coded as 0/1."
)

# Print the numbers used in the report
n_stays = analysis_df["RecordID"].nunique()
n_deaths = int(analysis_df[TARGET].sum())
death_rate = analysis_df[TARGET].mean()

print("=== Modeling table validation ===")
print(f"Rows before outcome check: {n_before}")
print(f"Rows excluded for missing outcome: {n_missing_outcome}")
print(f"Analyzable ICU stays: {n_stays}")
print(f"Unique RecordID: {analysis_df['RecordID'].nunique()}")
print(f"Deaths: {n_deaths}")
print(f"Death rate: {death_rate:.3%}")
print("\nOutcome counts:")
print(analysis_df[TARGET].value_counts().sort_index())

assert analysis_df["RecordID"].nunique() == len(analysis_df)
assert analysis_df[TARGET].nunique() == 2

- Verify that each row corresponds to one ICU stay, to avoid the same patient entering cross-validation multiple times.

- Missing outcomes cannot be treated as survival, so missing outcomes must not be automatically recoded as 0.

- Output the actual number and proportion of deaths, to provide the real results for the report.

In [ ]:
# Outcome
y = analysis_df[TARGET].astype(int)

# These columns must never be predictors
excluded_columns = [
    "RecordID",
    TARGET,
    "Length_of_stay",
    "Survival",
    "SAPS-I",
    "SOFA",
]

#  These categorical columns must not be interpreted as continuous numbers
categorical_candidates = ["Gender", "ICUType"]

# Predictors: only columns available in the modeling table
predictor_cols = [
    c for c in analysis_df.columns
    if c not in excluded_columns
]

X = analysis_df[predictor_cols].copy()

# Treat known categorical variables as categories
cat_cols = [c for c in categorical_candidates if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]

# Convert numeric predictors to numeric values, if necessary
for col in num_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

for col in cat_cols:
    X[col] = X[col].astype("category")

print("=== Predictor definition ===")
print(f"Number of candidate predictors: {X.shape[1]}")
print(f"Categorical predictors: {cat_cols}")
print(f"Numeric predictors: {len(num_cols)}")
print("Excluded columns that are present:",
      [c for c in excluded_columns if c in analysis_df.columns])
print("Predictor names:")
print(X.columns.tolist())

assert "RecordID" not in X.columns
assert TARGET not in X.columns
assert "Length_of_stay" not in X.columns
assert "Survival" not in X.columns
assert "SAPS-I" not in X.columns
assert "SOFA" not in X.columns

In [ ]:
# Missingness is evaluated at the stay level, after summarizing each stay.
missing_summary = pd.DataFrame({
    "missing_n": X.isna().sum(),
    "missing_pct": X.isna().mean() * 100,
    "observed_n": X.notna().sum(),
}).sort_values("missing_pct", ascending=False)

display(missing_summary.round(2))

# Save the table used in the report
missing_summary.to_csv(
    "output/cp2_predictor_missingness.csv",
    index=True
)

plt.figure(figsize=(10, 6))
top_missing = missing_summary.head(20).sort_values("missing_pct")
plt.barh(top_missing.index, top_missing["missing_pct"])
plt.xlabel("Missing stays (%)")
plt.ylabel("Predictor")
plt.title("Missingness among candidate predictors")
plt.tight_layout()
plt.savefig("output/cp2_missingness.png", dpi=200)
plt.show()

Missingness was examined for each candidate predictor. Missing measurements do not necessarily indicate normal physiology: a measurement may be absent because it was not ordered, not recorded, or not available. Measurement-count features were therefore retained as candidate predictors, subject to the same training-fold preprocessing as the other variables.

### Key observations
- Missingness varies substantially across the 190 candidate predictors.
- Troponin I and cholesterol summaries have the highest missingness, approximately 92–95%.
- Troponin T and respiratory rate summaries also show high missingness, approximately 72–78%.
- Albumin summaries have approximately 60% missingness.
- In contrast, variables such as Age, Gender, Height, ICUType, and pH_count have no missing values.

### Interpretation
- Many laboratory measurements are available for only a small subset of ICU stays, potentially reflecting differences in clinical testing practices.
- Highly incomplete predictors may contribute limited information and require careful consideration before inclusion in the models.
- Missingness should be handled within each cross-validation training fold to prevent data leakage.

## Set up 5-fold CV
Five-fold cross-validation divides the dataset into five subsets. It assesses predictive performance and stability on unseen data, reduces dependence on a single train-test split, and provides a consistent basis for comparing candidate models. The five-fold AUROC results and their variability will be summarized to guide final model selection.

In [ ]:
# 5-fold CV
SEED = 123
N_SPLITS = 5

outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

# Materialize folds once, then reuse the exact same splits for all models.
folds = list(outer_cv.split(X, y))

fold_rows = []
for fold_id, (train_idx, test_idx) in enumerate(folds, start=1):
    fold_rows.append({
        "fold": fold_id,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        "train_deaths": int(y.iloc[train_idx].sum()),
        "test_deaths": int(y.iloc[test_idx].sum()),
        "train_death_rate": float(y.iloc[train_idx].mean()),
        "test_death_rate": float(y.iloc[test_idx].mean()),
    })

fold_summary = pd.DataFrame(fold_rows)
display(fold_summary.round(3))
fold_summary.to_csv("output/cp2_fold_summary.csv", index=False)

assert len(folds) == N_SPLITS

### Key observations
- The dataset was divided into 5 folds, each containing 3,200 training stays and 800 test stays.
- Each test fold included 110–111 deaths, with a mortality rate of 13.8%–13.9%.
- Training mortality rates were similarly consistent at 13.8%–13.9%.
### Interpretation
- The similar mortality rates across folds indicate that stratified cross-validation successfully maintained a balanced distribution of the outcome.
- This consistency supports a fairer comparison of model performance across folds and reduces the likelihood that differences in mortality prevalence drive the results.

## Model 1: LASSO logistic regression
LASSO logistic regression uses an L1 penalty. It can shrink some coefficients to zero, making it useful for a wide table with many correlated summary features.

The regularization strength C is tuned in the inner cross-validation. In scikit-learn, smaller C means stronger regularization.

### Build pipelines

In [ ]:
# pipelines
def make_logistic_preprocessor(numeric_columns, categorical_columns):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_columns),
            ("cat", categorical_pipe, categorical_columns),
        ],
        remainder="drop",
    )


def make_tree_preprocessor(numeric_columns, categorical_columns):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        )),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_columns),
            ("cat", categorical_pipe, categorical_columns),
        ],
        remainder="drop",
    )


inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)


### Tune and evaluate

In [ ]:
lasso_pipeline = Pipeline([
    ("preprocess", make_logistic_preprocessor(num_cols, cat_cols)),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=5000,
        random_state=SEED
    )),
])

lasso_param_grid = {
    "model__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
}

oof = pd.DataFrame({
    "RecordID": analysis_df["RecordID"].to_numpy(),
    "y_true": y.to_numpy(),
})

fold_metrics = []
lasso_fold_models = []

for fold_id, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    search = GridSearchCV(
        estimator=clone(lasso_pipeline),
        param_grid=lasso_param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        refit=True,
        n_jobs=-1,
        error_score="raise",
    )

    search.fit(X_train, y_train)

    p_test = search.predict_proba(X_test)[:, 1]
    oof.loc[test_idx, "LASSO"] = p_test

    fold_metrics.append({
        "model": "LASSO",
        "fold": fold_id,
        "AUROC": roc_auc_score(y_test, p_test),
        "best_params": search.best_params_,
        "inner_cv_AUROC": search.best_score_,
    })

    lasso_fold_models.append(search.best_estimator_)

lasso_results = pd.DataFrame(fold_metrics)
display(lasso_results)
print("LASSO fold AUROC mean:",
      lasso_results["AUROC"].mean().round(4))
# 'float' object has no attribute 'round', use formatted output
print(f"LASSO fold AUROC SD:{lasso_results["AUROC"].std(ddof=1):.4f}")

#### Key observations
- The LASSO model achieved test-fold AUROCs ranging from 0.799 to 0.874 across the five folds.
- The highest test AUROC was 0.874 in Fold 1, while the lowest was 0.799 in Fold 4.
- The selected regularization parameter was C=0.1 in four folds and C=0.3 in Fold 1.
- Inner cross-validation AUROCs ranged from 0.823 to 0.848.
#### Interpretation
- The LASSO model demonstrated good discrimination overall, although performance varied across folds.
- The variation in selected C values suggests that the optimal regularization strength may depend somewhat on the training subset.
- Inner cross-validation was used to tune the hyperparameter, while the outer test folds provided estimates of out-of-sample performance. The outer-fold AUROCs should be used to compare models.

### Fit LASSO model
The first model is a logistic regression with an (L_1) penalty (LASSO). Logistic regression estimates the log odds of in-hospital mortality from the predictors, while the penalty discourages unnecessarily large coefficients and can set some coefficients to zero.

The penalty strength was selected by inner cross-validation within each outer training fold. Numeric predictors were median-imputed and standardized, categorical predictors were imputed and one-hot encoded, and missingness indicators were generated inside the pipeline.

AUROC was calculated for each held-out fold. The coefficient plot from the final refitted model will show which standardized predictors have the largest positive and negative associations with predicted mortality. Nonzero coefficients are predictive associations, not evidence that the corresponding variables cause mortality.

In [ ]:
# LASSO model
lasso_final_search = GridSearchCV(
    estimator=clone(lasso_pipeline),
    param_grid=lasso_param_grid,
    scoring="roc_auc",
    cv=inner_cv,
    refit=True,
    n_jobs=-1,
    error_score="raise",
)

lasso_final_search.fit(X, y)
lasso_final = lasso_final_search.best_estimator_

preprocessor = lasso_final.named_steps["preprocess"]
model = lasso_final.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
coef_values = model.coef_.ravel()

coef_table = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coef_values,
})
coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
coef_table = coef_table.sort_values(
    "abs_coefficient", ascending=False
)

display(coef_table.head(30).round(4))
coef_table.to_csv("output/cp2_lasso_coefficients.csv", index=False)

nonzero = coef_table.loc[coef_table["coefficient"] != 0].copy()
print("Selected nonzero coefficients:", len(nonzero))
display(nonzero[["feature", "coefficient"]].head(40).round(4))

plot_coef = coef_table.head(20).sort_values("coefficient")
plt.figure(figsize=(9, 7))
plt.barh(plot_coef["feature"], plot_coef["coefficient"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("LASSO coefficient (standardized numeric predictors)")
plt.ylabel("Predictor")
plt.title("Largest LASSO coefficients, refitted on all set-a")
plt.tight_layout()
plt.savefig("output/cp2_lasso_coefficients.png", dpi=200)
plt.show()

#### Key observation
- GCS and age have the largest coefficients in absolute value. GCS_last has the strongest negative coefficient (−0.809), while Age has the strongest positive coefficient (+0.474). Other prominent predictors include BUN_last (+0.267), Temp_mean (−0.261), and Bilirubin_last (+0.214).
- Several clinical measurements contribute to the model. BUN, bilirubin, glucose, lactate, heart rate, and multiple GCS summaries have nonzero coefficients, suggesting that renal function, liver function, metabolic status, and neurological status provide predictive information.
- Measurement frequency also matters in the fitted model. Features such as GCS_count, Urine_count, and Temp_count have nonzero coefficients. Missingness indicators for respiratory rate and mechanical ventilation also appear among the selected features.
- The model retains 125 nonzero coefficients after LASSO regularization, indicating that it selects a subset of predictors while shrinking others to zero.
#### Interpretation
- Clinical rationale: Age and BUN may capture baseline vulnerability and renal dysfunction, while bilirubin may reflect hepatic dysfunction. GCS summarizes neurological status, an important indicator of illness severity. These relationships are clinically plausible, but the coefficients alone do not establish causality.
- Positive coefficients indicate higher predicted log-odds of in-hospital death as the feature increases, conditional on the other model features; negative coefficients indicate lower predicted log-odds. However, the negative coefficients for GCS_last should not be interpreted as evidence on protective factors. Correlated predictors, missing-data imputation, and other model features may affect coefficient estimates.
- Measurement and missingness features may capture clinical practice, reflecting how frequently clinicians assess a patient or order particular tests and potentially providing additional information about illness severity and care patterns.
- Associations within the fitted predictive model cannot be interpreted as causal effects. The coefficients were estimated after refitting LASSO on all set-a data, so they help describe the final fitted model but should not replace the outer cross-validation results used to compare predictive performance.

### Examine selection stability across outer folds

In [ ]:
# stability across folds
fold_selection = []

for fold_id, fitted_pipeline in enumerate(lasso_fold_models, start=1):
    names = fitted_pipeline.named_steps["preprocess"].get_feature_names_out()
    coefs = fitted_pipeline.named_steps["model"].coef_.ravel()

    selected = set(names[coefs != 0])
    for name in names:
        fold_selection.append({
            "fold": fold_id,
            "feature": name,
            "selected": int(name in selected),
        })

selection_df = pd.DataFrame(fold_selection)
selection_stability = (
    selection_df.groupby("feature")["selected"]
    .agg(["sum", "mean"])
    .rename(columns={"sum": "folds_selected", "mean": "selection_fraction"})
    .sort_values("selection_fraction", ascending=False)
)

display(selection_stability.head(30).round(3))
selection_stability.to_csv(
    "output/cp2_lasso_selection_stability.csv"
)

print("Features selected in all five folds:",
      int((selection_stability["folds_selected"] == N_SPLITS).sum()))

#### Key observations
- 61 features were selected in all five cross-validation folds.
- The consistently selected features include clinical variables such as Age, BUN_last, Bilirubin_last, GCS_last, and HR_max, as well as numerous missingness indicators for laboratory tests and vital signs.
- Several features, including troponin and respiratory-rate missingness indicators, had a selection fraction of 1.0, meaning they were selected in all five folds.
#### Interpretation
- Stable feature selection suggests that these predictors repeatedly contributed to the LASSO model across different data splits, rather than being selected in only one particular fold.
- Variables such as age, renal and liver function markers, and GCS plausibly capture patient vulnerability and illness severity.
- Selection in all five folds does not prove that a feature is independently important or causally related to mortality. Missingness indicators may capture care processes or data-collection patterns.

## Model 2: non-linear effects with splines
The second model allows nonlinear associations between selected continuous predictors and mortality. The selected variables represent baseline characteristics, vital signs, neurological status, and laboratory measurements. The selection is based on clinical relevance and availability in the existing modeling table, not on the outcomes in the held-out folds.

A spline basis with a fixed number of knots is used for the selected continuous variables. Categorical variables remain categorical. The regularization strength is selected within the training data, and the same outer cross-validation folds are used as for Model 1.

For at least three continuous predictors, partial-effect curves with uncertainty bands will be examined. The curves describe adjusted predictive associations conditional on the other predictors in the fitted model; they should not be interpreted causally.

### Select predictors

In [ ]:
# spline regression
# Candidate continuous predictors chosen on clinical grounds.
# Names are matched to the summary columns that actually exist in X.
spline_candidates = [
    "Age",
    "HR_mean",
    "MAP_mean",
    "SysABP_mean",
    "DiasABP_mean",
    "Temp_mean",
    "RespRate_mean",
    "GCS_mean",
    "Glucose_mean",
    "Creatinine_mean",
    "BUN_mean",
    "Lactate_mean",
    "Na_mean",
    "K_mean",
    "WBC_mean",
    "Platelets_mean",
    "HCO3_mean",
    "PaO2_mean",
]

spline_cols = [c for c in spline_candidates if c in X.columns]

# Keep only numeric columns for spline transformations
spline_cols = [c for c in spline_cols if c in num_cols]

# Use ICUType/Gender if available, plus binary/state/count predictors
other_numeric_cols = [
    c for c in num_cols if c not in spline_cols
]
spline_cat_cols = cat_cols

print("Spline predictors available:", spline_cols)
print("Number of spline predictors:", len(spline_cols))
print("Other numeric predictors:", len(other_numeric_cols))
print("Categorical predictors:", spline_cat_cols)

assert len(spline_cols) >= 3, (
    "Fewer than three candidate continuous predictors were found. "
    "Inspect X.columns and revise spline_candidates."
)

# Keep the spline model manageable: use up to 16 continuous predictors.
spline_cols = spline_cols[:16]

X_spline = X[spline_cols + other_numeric_cols + spline_cat_cols].copy()

### Build & evaluate

In [ ]:
spline_numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("splines", SplineTransformer(
        n_knots=4,
        degree=3,
        include_bias=False
    )),
    ("scaler", StandardScaler()),
])

# For non-spline numeric predictors (e.g., binary indicators and counts),
# impute and scale without creating spline terms.
other_numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler()),
])

spline_categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

spline_preprocessor = ColumnTransformer(
    transformers=[
        ("spline", spline_numeric_pipe, spline_cols),
        ("other_num", other_numeric_pipe, other_numeric_cols),
        ("cat", spline_categorical_pipe, spline_cat_cols),
    ],
    remainder="drop",
)

spline_pipeline = Pipeline([
    ("preprocess", spline_preprocessor),
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=3000,
        random_state=SEED
    )),
])

spline_param_grid = {
    "model__C": [0.03, 0.1, 0.3, 1.0, 3.0]
}

spline_fold_models = []
spline_rows = []

for fold_id, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train = X_spline.iloc[train_idx]
    X_test = X_spline.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    search = GridSearchCV(
        estimator=clone(spline_pipeline),
        param_grid=spline_param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        refit=True,
        n_jobs=-1,
        error_score="raise",
    )

    search.fit(X_train, y_train)
    p_test = search.predict_proba(X_test)[:, 1]

    oof.loc[test_idx, "SplineLogistic"] = p_test

    spline_rows.append({
        "model": "SplineLogistic",
        "fold": fold_id,
        "AUROC": roc_auc_score(y_test, p_test),
        "best_params": search.best_params_,
        "inner_cv_AUROC": search.best_score_,
    })

    spline_fold_models.append(search.best_estimator_)

spline_results = pd.DataFrame(spline_rows)
display(spline_results)
print("Spline model AUROC mean:",
      spline_results["AUROC"].mean().round(4))
print(f"Spline model AUROC SD:{spline_results["AUROC"].std(ddof=1):.4f}")

#### Key observation
- The Spline Logistic model achieved a mean outer-fold AUROC of 0.8451 (SD = 0.0246) across five-fold cross-validation.
- Fold-level AUROCs ranged from 0.8046 to 0.8681, indicating some variation in predictive performance across folds.
- The optimal regularization parameter was consistently C=0.03 in all five folds.
- Inner-CV AUROC ranged from 0.8263 to 0.8365, slightly below the corresponding outer-fold AUROC in most folds.
#### Interpretation
- The mean AUROC of 0.8451 suggests good discrimination between patients who died in the hospital and those who survived.
- The consistent selection of C=0.03 suggests that the preferred regularization strength was stable across training subsets.
- The outer-fold AUROCs provide the primary estimates of out-of-sample performance. However, the relatively lower AUROC in Fold 4 indicates some variation in generalization across data splits.
- Although spline terms allow nonlinear predictor–outcome relationships, this model's performance alone does not establish that it outperforms simpler logistic regression because of noise and bias-variance trade-off.

### Fit spline model

In [ ]:
#final spline model
spline_final_search = GridSearchCV(
    estimator=clone(spline_pipeline),
    param_grid=spline_param_grid,
    scoring="roc_auc",
    cv=inner_cv,
    refit=True,
    n_jobs=-1,
    error_score="raise",
)

spline_final_search.fit(X_spline, y)
spline_final = spline_final_search.best_estimator_

print("Final spline model parameters:",
      spline_final_search.best_params_)
print("Final spline model inner-CV AUROC:",
      round(spline_final_search.best_score_, 4))

### Partial-effect curves
The following plots show conditional predicted log odds as each selected continuous variable changes while the other predictors are held at representative values. The bands use a large-sample covariance approximation for the fitted logistic regression coefficients; they are not bootstrap confidence intervals.

In [ ]:
# partial-effect curves
# Transform the full modeling data using the fitted preprocessing pipeline.
pre = spline_final.named_steps["preprocess"]
logit_model = spline_final.named_steps["model"]

Z = pre.transform(X_spline)
if hasattr(Z, "toarray"):
    Z = Z.toarray()
Z = np.asarray(Z, dtype=float)

beta = logit_model.coef_.ravel()
intercept = float(logit_model.intercept_[0])

# Approximate covariance matrix for logistic regression coefficients.
# This is a Wald approximation; the fitted model uses regularization.
p_full = expit(intercept + Z @ beta)
weights = np.clip(p_full * (1 - p_full), 1e-6, None)

design = np.column_stack([np.ones(len(Z)), Z])
beta_full = np.r_[intercept, beta]

information = design.T @ (weights[:, None] * design)
ridge = np.eye(information.shape[0]) * 1e-6
ridge[0, 0] = 0.0

cov_beta = np.linalg.pinv(information + ridge)

# Construct a representative profile: medians for numeric variables,
# modes for categorical variables.
reference = {}
for col in X_spline.columns:
    if col in spline_cat_cols:
        modes = X_spline[col].mode(dropna=True)
        reference[col] = modes.iloc[0] if len(modes) else np.nan
    else:
        reference[col] = X_spline[col].median()

reference_df = pd.DataFrame([reference])

plot_cols = spline_cols[:3]

for col in plot_cols:
    low = X_spline[col].quantile(0.05)
    high = X_spline[col].quantile(0.95)

    grid = np.linspace(low, high, 100)
    grid_df = pd.concat(
        [reference_df.copy() for _ in range(len(grid))],
        ignore_index=True
    )
    grid_df[col] = grid

    Z_grid = pre.transform(grid_df)
    if hasattr(Z_grid, "toarray"):
        Z_grid = Z_grid.toarray()
    Z_grid = np.asarray(Z_grid, dtype=float)

    D_grid = np.column_stack([np.ones(len(Z_grid)), Z_grid])
    eta = D_grid @ beta_full
    se_eta = np.sqrt(
        np.maximum(
            np.einsum("ij,jk,ik->i", D_grid, cov_beta, D_grid),
            0
        )
    )

    lower = eta - 1.96 * se_eta
    upper = eta + 1.96 * se_eta

    plt.figure(figsize=(7, 5))
    plt.plot(grid, eta, label="Conditional log odds")
    plt.fill_between(
        grid, lower, upper, alpha=0.2,
        label="Approximate 95% band"
    )
    plt.xlabel(col)
    plt.ylabel("Predicted log odds")
    plt.title(f"Partial effect of {col}")
    plt.legend()
    plt.tight_layout()
    
    safe_name = col.replace("/", "_").replace(" ", "_")
    plt.savefig(
        f"output/cp2_spline_effect_{safe_name}.png",
        dpi=200
    )
    plt.show()

#### Key observation
- All three partial-effect plots exhibit nearly flat estimated curves and unusually wide confidence bands, indicating substantial uncertainty in the estimated effects.
#### Interpretation
- Clinically, age is a plausible mortality risk factor, while heart rate and blood pressure may reflect cardiovascular status. However, these plots do not provide reliable evidence of the strength or shape of these associations because the confidence bands are exceptionally wide.

Although the curves appear to change very little, the substantial uncertainty means that these plots do not establish that the three variables have no association with in-hospital mortality.

In [ ]:
# Verify that the model's predictions remain almost constant across values of the variable.
from scipy.special import logit

reference = {}

for col in X_spline.columns:
    if col in spline_cat_cols:
        modes = X_spline[col].mode(dropna=True)
        reference[col] = modes.iloc[0] if len(modes) else np.nan
    else:
        reference[col] = X_spline[col].median()

reference_df = pd.DataFrame([reference])

diagnostic_rows = []

for col in spline_cols[:3]:
    low = X_spline[col].quantile(0.05)
    high = X_spline[col].quantile(0.95)

    grid = np.linspace(low, high, 100)

    grid_df = pd.concat(
        [reference_df.copy() for _ in range(len(grid))],
        ignore_index=True
    )
    grid_df[col] = grid

    # Use the fitted pipeline directly
    pred_prob = spline_final.predict_proba(grid_df)[:, 1]
    pred_log_odds = spline_final.decision_function(grid_df)

    diagnostic_rows.append({
        "feature": col,
        "grid_min": low,
        "grid_max": high,
        "probability_min": pred_prob.min(),
        "probability_max": pred_prob.max(),
        "probability_range": np.ptp(pred_prob),
        "log_odds_min": pred_log_odds.min(),
        "log_odds_max": pred_log_odds.max(),
        "log_odds_range": np.ptp(pred_log_odds),
    })

display(pd.DataFrame(diagnostic_rows).round(5))

All three variables indeed exhibited only small changes in both probability and log-odds.

In [ ]:
from scipy.special import expit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pre = spline_final.named_steps["preprocess"]
model = spline_final.named_steps["model"]

reference = {}
for col in X_spline.columns:
    if col in spline_cat_cols:
        modes = X_spline[col].mode(dropna=True)
        reference[col] = modes.iloc[0] if len(modes) else np.nan
    else:
        reference[col] = X_spline[col].median()

reference_df = pd.DataFrame([reference])

check_rows = []

for col in spline_cols[:3]:
    grid = np.linspace(
        X_spline[col].quantile(0.05),
        X_spline[col].quantile(0.95),
        100
    )

    grid_df = pd.concat(
        [reference_df.copy() for _ in grid],
        ignore_index=True
    )
    grid_df[col] = grid

    # no prediction
    prob = spline_final.predict_proba(grid_df)[:, 1]
    log_odds = model.decision_function(pre.transform(grid_df))

    check_rows.append({
        "feature": col,
        "probability_min": prob.min(),
        "probability_max": prob.max(),
        "probability_range": np.ptp(prob),
        "log_odds_min": log_odds.min(),
        "log_odds_max": log_odds.max(),
        "log_odds_range": np.ptp(log_odds)
    })

    # no bands, only curves
    plt.figure(figsize=(6, 4))
    plt.plot(grid, prob)
    plt.xlabel(col)
    plt.ylabel("Predicted probability")
    plt.title(f"Partial effect of {col}")
    plt.tight_layout()
    plt.show()

display(pd.DataFrame(check_rows).round(4))

The curve shows a certain trend, but it is obscured by the wide confidence intervals.

In [ ]:
# Check the exact transformed design matrix used by the fitted model
Z_model = spline_final.named_steps["preprocess"].transform(X_spline)

print("Input X_spline shape:", X_spline.shape)
print("Transformed matrix shape:", Z_model.shape)
print("Coefficient count:", spline_final.named_steps["model"].coef_.size)
print("Intercept count:",
      spline_final.named_steps["model"].intercept_.size)

assert Z_model.shape[1] == spline_final.named_steps["model"].coef_.size, (
    "Transformed feature count does not match model coefficient count."
)
# Check for near-constant columns and highly correlated features
Z_array = Z_model.toarray() if hasattr(Z_model, "toarray") else np.asarray(Z_model)

feature_sd = Z_array.std(axis=0)

print("Transformed feature matrix shape:", Z_array.shape)
print("Near-constant columns (SD < 1e-8):", (feature_sd < 1e-8).sum())
print("Minimum feature SD:", feature_sd.min())
print("Maximum feature SD:", feature_sd.max())

# Check the fitted logistic regression coefficients
coef = spline_final.named_steps["model"].coef_.ravel()

coef_summary = pd.Series(np.abs(coef)).describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)
print("\nAbsolute coefficient summary:")
print(coef_summary)

print("\nLargest absolute coefficients:")
print(
    pd.DataFrame({
        "feature_index": np.arange(len(coef)),
        "coefficient": coef
    })
    .assign(abs_coefficient=lambda d: d["coefficient"].abs())
    .nlargest(10, "abs_coefficient")
    .drop(columns="abs_coefficient")
)

In [ ]:
# Get transformed feature names
pre = spline_final.named_steps["preprocess"]
feature_names = pre.get_feature_names_out()

# Identify near-constant features
feature_sd = Z_array.std(axis=0)
constant_idx = np.where(feature_sd < 1e-8)[0]

constant_features = pd.DataFrame({
    "feature_index": constant_idx,
    "feature_name": feature_names[constant_idx],
    "standard_deviation": feature_sd[constant_idx]
})

print("Near-constant transformed features:")
display(constant_features)

# Check matrix rank and dimensionality
print("Matrix shape:", Z_array.shape)
print("Matrix rank:", np.linalg.matrix_rank(Z_array))
print("Number of columns:", Z_array.shape[1])

The primary reason for the extremely wide confidence band is the instability of the covariance estimates.

### New Partial-effect curves

In [ ]:
# new partial effect curve
from pathlib import Path

# Create output directory
Path("output").mkdir(parents=True, exist_ok=True)

# 1. Construct a representative patient profile
reference = {}

for col in X_spline.columns:
    if col in spline_cat_cols:
        modes = X_spline[col].mode(dropna=True)
        reference[col] = modes.iloc[0] if len(modes) else np.nan
    else:
        reference[col] = X_spline[col].median()

reference_df = pd.DataFrame([reference])

# 2. Plot conditional predicted probabilities
plot_cols = [
    col for col in ["Age", "HR_mean", "MAP_mean"]
    if col in X_spline.columns and col in spline_cols
]

for col in plot_cols:
    # Use the central 90% of observed values to avoid extreme extrapolation
    low = X_spline[col].quantile(0.05)
    high = X_spline[col].quantile(0.95)

    grid = np.linspace(low, high, 100)

    # Hold other predictors at their reference values
    grid_df = pd.concat(
        [reference_df.copy() for _ in range(len(grid))],
        ignore_index=True
    )
    grid_df[col] = grid

    # Predict through the complete fitted pipeline
    predicted_prob = spline_final.predict_proba(grid_df)[:, 1]

    # Plot
    plt.figure(figsize=(7, 5))
    plt.plot(grid, predicted_prob, linewidth=2)

    plt.xlabel(col)
    plt.ylabel("Predicted probability of in-hospital death")
    plt.title(f"Conditional partial effect of {col}")
    plt.ylim(bottom=0)
    plt.tight_layout()

    safe_name = col.replace("/", "_").replace(" ", "_")
    plt.savefig(
        f"output/newc_cp2_spline_effect_{safe_name}.png",
        dpi=200,
        bbox_inches="tight"
    )
    plt.show()

    print(
        f"{col}: predicted probability range = "
        f"{predicted_prob.min():.4f} to {predicted_prob.max():.4f}; "
        f"range width = {np.ptp(predicted_prob):.4f}"
    )

#### Key observation

- Predicted in-hospital mortality increases monotonically with age. The predicted probability increases from 1.6% to 5.8% (range width = 4.15 percentage points).
- The relationship in HR_mean is U-shaped. Predicted probability ranges from 2.86% to 5.00% (range width = 2.14 percentage points).
- Predicted mortality changes only minimally across the observed MAP range. The curve shows a slight upward trend, but the overall probability range is narrow.

#### Interpretation (with caution since the confidence intervals are wide)

- The association appears nonlinear, with risk accelerating at older ages, suggesting the effect of age becomes stronger in elderly patients.
- Moderate heart rates are associated with the lowest predicted mortality. Elevated heart rate is associated with increased risk, while very low heart rates may also be linked to somewhat higher risk, indicating a nonlinear effect.
- MAP_mean appears to have a weak marginal effect on predicted mortality in this model.

## Model 3: tree ensemble (random forest)
The third model is a random forest classifier. Random forests combine many decision trees and can represent nonlinear effects and interactions without specifying them in advance.

The tuning grid varies maximum tree depth and minimum leaf size. Tuning occurs within the training portion of each outer fold. Numeric predictors are median-imputed and categorical predictors are imputed and one-hot encoded; no scaling is required for the tree model.

The same outer folds used for the logistic and spline models are used here. AUROC is reported for every held-out fold so that the performance of the three models can be compared under the same validation design.

### Tune and evaluate

In [ ]:
# random forest
rf_pipeline = Pipeline([
    ("preprocess", make_tree_preprocessor(num_cols, cat_cols)),
    ("model", RandomForestClassifier(
        n_estimators=150,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1,
    )),
])

rf_param_grid = {
    "model__max_depth": [5, None],
    "model__min_samples_leaf": [5, 15],
}

rf_fold_models = []
rf_rows = []

for fold_id, (train_idx, test_idx) in enumerate(folds, start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    search = GridSearchCV(
        estimator=clone(rf_pipeline),
        param_grid=rf_param_grid,
        scoring="roc_auc",
        cv=inner_cv,
        refit=True,
        n_jobs=1,  # Avoid nested parallelism with the forest's n_jobs=-1
        error_score="raise",
    )

    search.fit(X_train, y_train)
    p_test = search.predict_proba(X_test)[:, 1]

    oof.loc[test_idx, "RandomForest"] = p_test

    rf_rows.append({
        "model": "RandomForest",
        "fold": fold_id,
        "AUROC": roc_auc_score(y_test, p_test),
        "best_params": search.best_params_,
        "inner_cv_AUROC": search.best_score_,
    })

    rf_fold_models.append(search.best_estimator_)

rf_results = pd.DataFrame(rf_rows)
display(rf_results)
print("Random forest AUROC mean:",
      rf_results["AUROC"].mean().round(4))
print(f"Random forest AUROC SD:{rf_results["AUROC"].std(ddof=1):.4f}")

#### Key observations
- The Random Forest model achieved a mean outer-fold AUROC of 0.8428 (SD = 0.0168) across five-fold cross-validation.
- Fold-level AUROCs ranged from 0.8203 to 0.8667, with the highest performance in Fold 5 and the lowest in Fold 4.
- Inner-CV AUROCs ranged from 0.8323 to 0.8509.

#### Interpretation
- The mean AUROC indicates good discrimination between patients who died in the hospital and those who survived.
- The relatively small standard deviation suggests reasonably consistent performance across folds, although Fold 4 showed weaker discrimination.
- The outer-fold AUROCs provide the primary estimates of generalization performance; the inner-CV AUROCs were used for hyperparameter tuning.

### Fit random forest model

In [ ]:
# final one
rf_final_search = GridSearchCV(
    estimator=clone(rf_pipeline),
    param_grid=rf_param_grid,
    scoring="roc_auc",
    cv=inner_cv,
    refit=True,
    n_jobs=1,
    error_score="raise",
)

rf_final_search.fit(X, y)
rf_final = rf_final_search.best_estimator_

print("Final random forest parameters:",
      rf_final_search.best_params_)
print("Final random forest inner-CV AUROC:",
      round(rf_final_search.best_score_, 4))

rf_feature_names = (
    rf_final.named_steps["preprocess"].get_feature_names_out()
)
rf_importances = rf_final.named_steps["model"].feature_importances_

rf_importance_table = pd.DataFrame({
    "feature": rf_feature_names,
    "importance": rf_importances,
}).sort_values("importance", ascending=False)

display(rf_importance_table.head(25).round(4))
rf_importance_table.to_csv(
    "output/cp2_random_forest_importance.csv",
    index=False
)

top_rf = rf_importance_table.head(20).sort_values("importance")
plt.figure(figsize=(9, 7))
plt.barh(top_rf["feature"], top_rf["importance"])
plt.xlabel("Mean decrease in impurity importance")
plt.ylabel("Predictor")
plt.title("Random forest feature importance")
plt.tight_layout()
plt.savefig("output/cp2_random_forest_importance.png", dpi=200)
plt.show()

The Random Forest feature-importance analysis identified:
 
- The last GCS score, cumulative urine output, mean GCS score, and mean urine output as the most important predictors.
- Several BUN-related features also ranked highly, suggesting that measures of neurological status, urine output, and renal function provided substantial predictive information in the fitted model.
- In contrast, mean heart rate, temperature, and several other physiological features had comparatively lower importance. 

These results reflect the relative contributions of features to the Random Forest prediction process and should not be interpreted as evidence of effect direction, independent associations, or causality.

## Comparison of final model
SAPS-I and SOFA are existing clinical severity scores. In this project, they should be evaluated separately as benchmark scores, rather than as model predictors.

To ensure a fair comparison, we use the same set of outer test folds. For each fold, we separately calculate:

- the AUROC of LASSO logistic regression;
- the AUROC of spline logistic regression;
- the AUROC of random forest;
- the AUROC of SAPS-I when used alone as a risk score;
- the AUROC of SOFA when used alone as a risk score.

### Build dataframe for benchmark

In [ ]:
# using “SAPS-I”, "SOFA"
model2_df = model_df.copy()
model2_df = feature_df.merge(
    outcome2_df,
    on="RecordID",
    how="left",
    validate="one_to_one",
    indicator=True
)
analysis2_df = model2_df.loc[
    model2_df["death"].notna()
].copy()

benchmark_candidates = ["SAPS-I", "SOFA"]
benchmark_cols = [
    col for col in benchmark_candidates
    if col in analysis2_df.columns
]

print("Available benchmark scores:", benchmark_cols)

### Inspect SAPS-I & SOFA

In [ ]:
# check "SAPS-I", "SOFA"
# Inspect the key columns in each dataset

check_cols = [
    c for c in [
        "death",
        "SAPS-I",
        "SOFA",
        "Length_of_stay",
        "Survival",
    ]
    if c in outcome2_df.columns and c in model2_df.columns
]

comparison = model2_df[["RecordID"] + check_cols].merge(
    outcome2_df[["RecordID"] + check_cols],
    on="RecordID",
    how="outer",
    suffixes=("_model", "_outcome"),
    indicator=True,
    validate="one_to_one",
)

print(comparison["_merge"].value_counts(dropna=False))

for col in check_cols:
    a = comparison[f"{col}_model"]
    b = comparison[f"{col}_outcome"]

    same = a.eq(b) | (a.isna() & b.isna())
    print(f"{col}: mismatches =", int((~same).sum()))

benchmark_cols = ["SAPS-I", "SOFA"]
benchmark_cols = [
    c for c in benchmark_cols if c in analysis2_df.columns
]

print("Analysis cohort size:", len(analysis2_df))

for col in benchmark_cols:
    s = analysis2_df[col]
    print(f"\n--- {col} ---")
    print("Data type:", s.dtype)
    print("Missing:", s.isna().sum())
    print("Missing proportion:", round(s.isna().mean(), 4))
    print("Unique values:", s.nunique(dropna=True))
    print(s.describe())

benchmark_rows = []

for fold_id, (_, test_idx) in enumerate(folds, start=1):
    fold_df = analysis2_df.iloc[test_idx]
    y_test = fold_df["death"]

    for col in benchmark_cols:
        scores = fold_df[col]

        # Only evaluate rows with observed outcome and score
        valid = y_test.notna() & scores.notna()
        y_valid = y_test.loc[valid]
        score_valid = scores.loc[valid]

        if len(y_valid) == 0 or y_valid.nunique() < 2:
            auc = np.nan
        else:
            auc = roc_auc_score(y_valid, score_valid)

        benchmark_rows.append({
            "benchmark": col,
            "fold": fold_id,
            "n_evaluated": int(valid.sum()),
            "n_deaths": int((y_valid == 1).sum()),
            "n_survivors": int((y_valid == 0).sum()),
            "AUROC": auc,
        })

benchmark_check = pd.DataFrame(benchmark_rows)
display(benchmark_check.round(4))

In [ ]:
display(
    benchmark_check.groupby("benchmark").agg(
        total_evaluated=("n_evaluated", "sum"),
        total_deaths=("n_deaths", "sum"),
        total_survivors=("n_survivors", "sum"),
        folds_with_auc=("AUROC", "count"),
    )
)

No mismatches, missing values, or imbalanced sample sizes across folds.

In [ ]:
for col in benchmark_cols:
    s = analysis2_df[col]

    print(f"\n--- {col} ---")
    print("Python dtype:", s.dtype)
    print("Example values:")
    print(s.dropna().head(10).tolist())

    numeric = pd.to_numeric(s, errors="coerce")
    invalid = s.notna() & numeric.isna()

    print("Non-numeric values:", int(invalid.sum()))

No incorrectly transformation or overwritten.

In [ ]:
target = "death"
y = analysis2_df[target]

print("Target dtype:", y.dtype)
print("\nValue counts (including missing):")
print(y.value_counts(dropna=False).sort_index())

print("\nMissing outcomes:", y.isna().sum())
print("Unique non-missing values:", sorted(y.dropna().unique()))

valid_y = y.dropna()

if set(valid_y.unique()).issubset({0, 1}):
    print("Death count:", int((valid_y == 1).sum()))
    print("Survivor count:", int((valid_y == 0).sum()))
    print("Death rate:", round((valid_y == 1).mean(), 4))
else:
    print("WARNING: Outcome is not coded exclusively as 0 and 1.")

# Compare the target values after merging with the outcome source
target_check = model2_df[["RecordID", target]].merge(
    outcome2_df[["RecordID", target]],
    on="RecordID",
    how="outer",
    suffixes=("_model", "_source"),
    indicator=True,
    validate="one_to_one",
)

a = target_check[f"{target}_model"]
b = target_check[f"{target}_source"]

same = a.eq(b) | (a.isna() & b.isna())

print("Merge status:")
print(target_check["_merge"].value_counts(dropna=False))
print("\nOutcome mismatches:", int((~same).sum()))

Correctly code "death", no mismatches.

In [ ]:
for col in benchmark_cols:
    valid = (
        analysis2_df["death"].notna()
        & analysis2_df[col].notna()
    )

    y_valid = analysis2_df.loc[valid, "death"]
    score_valid = analysis2_df.loc[valid, col]

    if y_valid.nunique() == 2:
        auc = roc_auc_score(y_valid, score_valid)
        print(
            f"{col}: AUROC={auc:.4f}, "
            f"n={int(valid.sum())}, "
            f"deaths={int((y_valid == 1).sum())}"
        )

Based on the above checks, SAPS-I and SOFA have relatively low AUROC values, which reflects the performance of the data itself.

### Evaluate benchmark scores

In [ ]:
# compare the three models with SAPS-I and SOFA
benchmark_candidates = ["SAPS-I", "SOFA"]
benchmark_cols = [
    c for c in benchmark_candidates
    if c in analysis2_df.columns
]

print("Available benchmark scores:", benchmark_cols)

for col in benchmark_cols:
    analysis2_df[col] = pd.to_numeric(
        analysis2_df[col], errors="coerce"
    )
    oof[col] = analysis2_df[col].to_numpy()

benchmark_rows = []

for fold_id, (_, test_idx) in enumerate(folds, start=1):
    y_test = y.iloc[test_idx]

    for score_name in benchmark_cols:
        scores = analysis2_df.iloc[test_idx][score_name]
        valid = scores.notna()

        if y_test.loc[valid.index[valid]].nunique() < 2:
            fold_auc = np.nan
        elif valid.sum() == 0 or y_test.loc[valid.index[valid]].nunique() < 2:
            fold_auc = np.nan
        else:
            fold_auc = roc_auc_score(
                y_test.loc[valid.index[valid]],
                scores.loc[valid.index[valid]]
            )

        benchmark_rows.append({
            "model": score_name,
            "fold": fold_id,
            "AUROC": fold_auc,
            "n_evaluated": int(valid.sum()),
        })

benchmark_results = pd.DataFrame(benchmark_rows)
display(benchmark_results)

### Output result table

In [ ]:
# result table
all_fold_results = pd.concat(
    [
        lasso_results[["model", "fold", "AUROC"]],
        spline_results[["model", "fold", "AUROC"]],
        rf_results[["model", "fold", "AUROC"]],
        benchmark_results[["model", "fold", "AUROC"]],
    ],
    ignore_index=True
)

comparison_table = (
    all_fold_results.groupby("model")["AUROC"]
    .agg(mean_AUROC="mean", sd_AUROC="std", folds_evaluated="count")
    .reset_index()
    .sort_values("mean_AUROC", ascending=False)
)

display(comparison_table.round(4))
comparison_table.to_csv(
    "output/cp2_model_comparison.csv",
    index=False
)

print("Fold-level AUROC results:")
display(all_fold_results.sort_values(["model", "fold"]).round(4))
all_fold_results.to_csv(
    "output/cp2_fold_auroc.csv",
    index=False
)

#### Key observations

- SplineLogistic achieved the highest mean AUROC (0.8451, SD = 0.0246), followed closely by LASSO (0.8430, SD = 0.0292) and RandomForest (0.8428, SD = 0.0168).
- All three models outperformed the clinical benchmarks, SAPS-I (0.6379) and SOFA (0.6262), by a substantial margin.
- The three predictive models showed some fold-to-fold variation; Fold 4 had the lowest AUROC for all three.

#### Interpretation

- The similar AUROCs suggest that flexible nonlinear modeling and RandomForest provide little improvement in discrimination over LASSO in this evaluation.
- RandomForest had the lowest fold-level SD among the three predictive models, suggesting more consistent performance across folds.
- Overall, the models demonstrated good discrimination in cross-validation, but these results alone do not establish calibration, clinical utility, or external generalizability.

### Plot out-of-fold ROC curves

In [ ]:
# plot out-of-fold ROC curves
# OOF predictions from the three fitted models
prediction_cols = ["LASSO", "SplineLogistic", "RandomForest"]

# Add benchmark columns if available
prediction_cols += [
    c for c in benchmark_cols if c in oof.columns
]

plt.figure(figsize=(8, 7))

for col in prediction_cols:
    valid = oof[col].notna()
    if valid.sum() == 0 or oof.loc[valid, "y_true"].nunique() < 2:
        continue

    fpr, tpr, _ = roc_curve(
        oof.loc[valid, "y_true"],
        oof.loc[valid, col]
    )
    score = roc_auc_score(
        oof.loc[valid, "y_true"],
        oof.loc[valid, col]
    )

    plt.plot(fpr, tpr, label=f"{col} (AUROC = {score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Out-of-fold ROC curves on set-a")
plt.legend()
plt.tight_layout()
plt.savefig("output/cp2_oof_roc.png", dpi=200)
plt.show()

# Save predictions for later comparison and reproducibility
oof.to_csv("output/cp2_oof_predictions.csv", index=False)

The three fitted models were compared with SAPS-I and SOFA, each evaluated as a standalone risk score. The comparison table reports the AUROC in each outer fold, the mean AUROC, and the standard deviation across folds. Out-of-fold ROC curves provide a visual comparison using predictions generated only when each stay was held out from model fitting.

A model with a higher mean AUROC has better ranking discrimination in these folds, but differences should be interpreted relative to fold-to-fold variability. A small numerical difference does not necessarily establish a meaningful improvement. Missing benchmark scores, if any, are evaluated only among stays with observed scores and are reported transparently.

The final model is selected by considering discrimination, stability, complexity, and interpretability rather than choosing a model solely because it has the largest estimated AUROC. Checkpoint 2 does not require calibration curves, decision curve analysis, subgroup analyses, or SHAP-style explanations; those belong to Checkpoint 3.

## Final model selection
After reviewing comparison_table, choose LASSO as the final model.

Although its mean AUROC is not the highest, given the model's simplicity and interpretability — and when performance is nearly identical — a sparse logistic regression is easier to interpret and better suited as a clear statistical modeling benchmark.

In [ ]:
# After reviewing comparison_table, choose the final model.
# Allowed values: "LASSO", "SplineLogistic", "RandomForest"
FINAL_MODEL_NAME = "LASSO"

if FINAL_MODEL_NAME == "LASSO":
    final_pipeline = lasso_final
    final_features = predictor_cols
elif FINAL_MODEL_NAME == "SplineLogistic":
    final_pipeline = spline_final
    final_features = X_spline.columns.tolist()
elif FINAL_MODEL_NAME == "RandomForest":
    final_pipeline = rf_final
    final_features = predictor_cols
else:
    raise ValueError(
        "FINAL_MODEL_NAME must be LASSO, SplineLogistic, or RandomForest."
    )

# Save the fitted pipeline, including its fitted preprocessing.
final_artifact = {
    "model_name": FINAL_MODEL_NAME,
    "pipeline": final_pipeline,
    "feature_columns": final_features,
    "target": TARGET,
    "seed": SEED,
}

joblib.dump(
    final_artifact,
    "output/cp2_final_model.joblib"
)

# Save fold-level results and OOF predictions
all_fold_results.to_csv(
    "output/cp2_fold_auroc.csv",
    index=False
)
comparison_table.to_csv(
    "output/cp2_model_comparison.csv",
    index=False
)
oof.to_csv(
    "output/cp2_oof_predictions.csv",
    index=False
)

print("Final model saved to output/cp2_final_model.joblib")
print("Selected model:", FINAL_MODEL_NAME)
print("Predictor count:", len(final_features))
print("Saved OOF predictions:", "output/cp2_oof_predictions.csv")

The final model was selected after comparing the three candidate models and the SAPS-I and SOFA benchmarks using the same outer cross-validation folds. The decision considered mean AUROC, variation across folds, model complexity, and interpretability.

The selected model and its fitted preprocessing steps were saved to output/cp2_final_model.joblib. Out-of-fold predictions for the candidate models and benchmarks were saved to output/cp2_oof_predictions.csv, and fold-level AUROC results were saved to output/cp2_fold_auroc.csv.

The model was fitted using set-a only. The saved pipeline preserves the preprocessing learned from set-a and is intended to be applied without refitting or retuning when evaluating the model on the held-out set-b and set-c data in Checkpoint 3.

## Final check

In [ ]:
# final check
# Check that all three models have five outer-fold AUROCs
expected_models = {"LASSO", "SplineLogistic", "RandomForest"}

for name, result in [
    ("LASSO", lasso_results),
    ("SplineLogistic", spline_results),
    ("RandomForest", rf_results),
]:
    assert result["fold"].nunique() == N_SPLITS, (
        f"{name} does not have results for all outer folds."
    )
    assert result["AUROC"].notna().all(), (
        f"{name} has missing fold AUROC values."
    )

# Check that every ICU stay has one out-of-fold prediction per fitted model
for col in ["LASSO", "SplineLogistic", "RandomForest"]:
    assert oof[col].notna().all(), (
        f"Missing OOF predictions for {col}."
    )

# Verify output files
required_files = [
    "output/cp2_final_model.joblib",
    "output/cp2_oof_predictions.csv",
    "output/cp2_fold_auroc.csv",
    "output/cp2_model_comparison.csv",
    "output/cp2_lasso_coefficients.csv",
    "output/cp2_lasso_selection_stability.csv",
    "output/cp2_random_forest_importance.csv",
    "output/cp2_oof_roc.png",
]

print("=== Final validation ===")
print("Unique ICU stays:", analysis_df["RecordID"].nunique())
print("Outcome missing before exclusion:", n_missing_outcome)
print("Deaths:", int(y.sum()))
print("Death rate:", f"{y.mean():.3%}")
print("All fitted models evaluated on the same outer folds:", True)
print("All three models have complete OOF predictions:", True)

print("\nRequired output files:")
for path in required_files:
    print(f"{path}: {'OK' if os.path.exists(path) else 'MISSING'}")

print("\nFinal model:", FINAL_MODEL_NAME)
print("\nFinal comparison table:")
display(comparison_table.round(4))

However, the analysis uses a fixed random seed and reuses the same outer cross-validation folds for all candidate models and benchmark scores. The fitted final pipeline, out-of-fold predictions, fold-level AUROC results, model comparison table, and model interpretation outputs are saved under output/.

Several limitations should be considered. First, the analysis uses only set-a, so cross-validation estimates internal discrimination rather than external generalizability. Second, missing measurements can reflect both patient condition and clinical monitoring decisions. Third, summaries such as means, minima, maxima, and last observed values simplify time-varying physiology and may lose information about timing or trajectories. Fourth, variable importance and regression coefficients describe predictive associations and should not be interpreted as causal effects. Fifth, The Wald intervals based on the estimated covariance matrix was unstable in this model.

The final model should be evaluated on set-b and set-c only in Checkpoint 3. Those datasets must not be used to retune hyperparameters or select the model.